In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *
import pyreadr
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
#######
#Code for QC, normalizing and hvg over all interventions
#######

In [ ]:
adata = import_data()

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata_raw = adata.copy()

In [ ]:
adata = adata_raw.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.var["gene_short_name"] = adata.var["gene_short_name"].astype(str)
adata.var_names = adata.var["gene_short_name"].values
adata.var_names_make_unique()

In [ ]:
targets = list(adata.obs['gene_target'].unique())
hvg = pd.Series(False, index=adata.var_names)

for gene_target in targets:
    print(gene_target)
    adata_paired = adata[(adata.obs['gene_target']=='ctrl-inj') | (adata.obs['gene_target']==gene_target)]
    sc.pp.highly_variable_genes(adata_paired, n_top_genes=2000)
    hvg = hvg | adata_paired.var['highly_variable']
    
adata.var['highly_variable'] = hvg

In [ ]:
hvg.value_counts()

In [3]:
path = "/home/azweig/projects/zebrafish/data"
suffix = "pairwise_hvg.h5ad"
filename = os.path.join(path, suffix)
# adata.write(filename)

In [4]:
#################################################

In [5]:
adata = load_data(filename)

In [6]:
list(adata.obs['gene_target'].unique())

['ctrl-inj',
 'ctrl-noto',
 'ctrl-met',
 'ctrl-hgfa',
 'ctrl-mafba',
 'ctrl-tbx16',
 'zc4h2',
 'met',
 'tfap2a',
 'hgfa-mut',
 'tfap2a-foxd3',
 'noto',
 'cdx4-cdx1a',
 'epha4a',
 'mafba',
 'tbx16-msgn1',
 'smo',
 'hand2',
 'egr2b',
 'noto-mut',
 'cdx4',
 'foxi1',
 'hoxb1a',
 'tbx16-tbx16l',
 'mafba-mut',
 'tbx16',
 'tbx1',
 'wnt3a-wnt8',
 'phox2a',
 'foxd3',
 'met-mut',
 'hgfa',
 'tbxta',
 'tbx16-mut']

In [12]:
adata

View of AnnData object with n_obs × n_vars = 2686378 × 4569
    obs: 'cell', 'Size_Factor', 'n.umi', 'perc_mitochondrial_umis', 'timepoint', 'hash_umis', 'top_to_second_best_ratio', 'expt', 'cell_type_sub', 'cell_type_broad', 'tissue', 'germ_layer', 'log.n.umi', 'num_genes_expressed', 'umap3d_1', 'umap3d_2', 'umap3d_3', 'major_group', 'gene_target', 'mean_nn_time', 'subumap3d_1', 'subumap3d_2', 'subumap3d_3', 'embryo', 'temp', 'n_genes'
    var: 'gene_short_name', 'id', 'chromosome', 'bp1', 'bp2', 'gene_strand', 'num_cells_expressed', 'n_cells', 'highly_variable'
    uns: 'log1p'

In [ ]:
gene_target='tbx16-tbx16l'
adata_local = adata[adata.obs['gene_target'] == gene_target]
sc.tl.pca(adata_local)
sc.pp.neighbors(adata_local)
sc.tl.umap(adata_local)

fig, (ax1, ax2) = plt.subplots(2, figsize=(16, 16))
sc.pl.umap(
    adata_local,
    color="tissue",
    size=2,
    ax=ax1,
    show=False
)
sc.pl.umap(adata_local, color="timepoint", size=2, ax=ax2, show=False)

In [ ]:
def plot_perturb(gene_target='tbx16-tbx16l', t=[36]):
    adata_local = adata[adata.obs['timepoint'].isin(t)]
    if adata_local[adata_local.obs['gene_target'] == gene_target].shape[0] == 0:
        print("empty perturbation")
        return
        
    adata_local = adata_local[(adata_local.obs['gene_target'] == 'ctrl-inj') | (adata_local.obs['gene_target'] == gene_target)]
    sc.tl.pca(adata_local)
    sc.pp.neighbors(adata_local)
    sc.tl.umap(adata_local)

    fig, (ax1, ax2) = plt.subplots(2, figsize=(16, 16))

    fig.suptitle(f'Control versus {gene_target} at times {t}')
    
    sc.pl.umap(
        adata_local,
        color="tissue",
        size=2,
        ax=ax1,
        show=False
    )
    
    sc.pl.umap(adata_local, color="gene_target", size=2, ax=ax2, show=False)
    
    plt.show()


In [ ]:
for gene_target in list(adata.obs['gene_target'].unique()):
    plot_perturb(gene_target)

In [ ]:
###############

In [3]:
path = "/home/azweig/projects/zebrafish/data"
suffix = "pairwise_hvg.h5ad"
filename = os.path.join(path, suffix)
adata = load_data(filename)

In [4]:
sc.tl.pca(adata, n_comps = 50, mask_var = None)

/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:379: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  adata.obsm[key_obsm] = X_pca


In [6]:
import sys
import os
sys.path.append(os.path.abspath("conditional-flow-matching"))
from torchcfm.optimal_transport import OTPlanSampler

In [7]:
batch_size = 512
t0 = 18
t1 = 36
ot_sampler = OTPlanSampler(method="exact")

In [9]:
# adata.obs.keys()

In [14]:
def get_labeled_samples(batch_size, t):
    adata_local = adata[(adata.obs['gene_target'] == 'ctrl-inj') & (adata.obs['timepoint'] == t)]
    X = adata_local.obsm['X_pca']
    indices = np.random.choice(X.shape[0], batch_size, replace=False)
    x = X[indices]
    y = adata_local.obs['cell_type_broad']
    return torch.from_numpy(x), y

In [15]:
x0, y0 = get_labeled_samples(batch_size, t0)
x1, y1 = get_labeled_samples(batch_size, t1)

In [16]:
x0, x1, y0, y1 = ot_sampler.sample_plan_with_labels(x0, x1, y0, y1)

/home/azweig/projects/zebrafish/conditional-flow-matching/torchcfm/optimal_transport.py:180: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y0[i] if y0 is not None else None,
/home/azweig/projects/zebrafish/conditional-flow-matching/torchcfm/optimal_transport.py:181: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y1[j] if y1 is not None else None,


In [25]:
max_left_width = 46

aligned_lines = [
    f"{y0[i]:<{max_left_width}} --> {y1[i]}"
    for i in range(20)
]

for line in aligned_lines:
    print(line)

fin mesenchyme                                 --> posterior spinal cord progenitors
eye, optic cup                                 --> head mesenchyme/PA cartilage
periderm                                       --> differentiating neuron 2
head/eye connective tissue                     --> neural progenitor (MHB)
pharyngeal arch (NC-derived)                   --> mature fast muscle
fast-committed myocyte                         --> periderm
connective tissue-meninges-dermal FB           --> neurons (gabaergic, glutamatergic; contains Purkinje)
neural progenitor (MHB)                        --> periderm
mature slow muscle                             --> hatching gland
fast-committed myocyte                         --> basal cell
endothelium (vein + early artery)              --> periderm
periderm                                       --> periderm
neural progenitor (telencephalon/diencephalon) --> connective tissue-meninges-dermal FB
neural progenitor (telencephalon/diencephalon) --> cr

/tmp/ipykernel_1925378/2388599626.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  f"{y0[i]:<{max_left_width}} --> {y1[i]}"
